In [21]:
pip install lammps-interface

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [22]:
import os, shutil, math, glob
from lammps_interface.lammps_main import LammpsSimulation
from lammps_interface.structure_data import from_CIF, write_CIF
import pandas as pds
import numpy as np
import sys

sys.setrecursionlimit(100000)


import os, shutil, math, glob

parent_dir = '/storage/home/hcoda1/5/sjamdade3/scratch/August_13_2023/water_screening/MD_MC/Manuscript_Final/Defects/'

path_general_files = '/storage/home/hcoda1/5/sjamdade3/scratch/August_13_2023/water_screening/MD_MC/Manuscript_Final/Defects/SOP_files/'

MOF_type = 'defects/'

In [23]:
from array import *
def array_list(array_num):
    num_list = array_num.tolist() # list

In [24]:
class Parameters:
    def __init__(self, cif):
        # File options
        self.cif_file = cif
        self.output_cif = False
        self.output_raspa = False

        # Force field options
        self.force_field = 'UFF'
        self.mol_ff = False
        self.h_bonding = False
        self.dreid_bond_type = 'harmonic'
        self.fix_metal = False
        
        # Simulation options
        self.minimize = False
        self.bulk_moduli = False
        self.thermal_scaling = False
        self.npt = False
        self.nvt = False
        self.cutoff = 10.0
        self.replication = None
        self.orthogonalize = False
        self.random_vel = True
        self.dump_dcd = 0
        self.dump_xyz = 0
        self.dump_lammpstrj = 0
        self.restart = False
        
        # Parameter options
        self.tol = 0.4
        self.neighbour_size = 5
        self.iter_count = 10
        self.max_dev = 0.01
        self.temp = 298.0
        self.pressure = 0.00740192
        self.nprodstp = 200000
        self.neqstp = 200000
        
        # Molecule insertion options
        self.insert_molecule = ''
        self.deposit = 0
        
    def show(self):
        for v in vars(self):
            print('%-15s: %s' % (v, getattr(self, v)))

In [25]:
##### LAMMPS NVT MD 

path_CIFs = os.path.join(parent_dir, 'MOFs', str(MOF_type))

path_MOF_LAMMPS_NVT_MD = os.path.join(path_CIFs, 'LAMMPS_NVT_MD/')

os.mkdir(path_MOF_LAMMPS_NVT_MD)

path_LAMMPS_NVT_MD_input_files = os.path.join(path_general_files, 'LAMMPS_NVT_MD_input_files/')

path_MOF_NVT_MC = os.path.join(path_CIFs, 'NVT_MC/')

files_CIFs = os.listdir(path_CIFs)
        

In [26]:
### UFF Parameters
LJ_UFF = os.path.join(path_LAMMPS_NVT_MD_input_files, 'UFF_data.xlsx')
LJ_data = pds.read_excel(LJ_UFF, 'LJ' )
ELEMENT = np.array(LJ_data['Element'])
ELEMENT = ELEMENT.tolist()

EPSILON = np.array(LJ_data['Energy(kcal/mol) '])
EPSILON = EPSILON.tolist()

SIGMA = np.array(LJ_data['sigma'])
SIGMA = SIGMA.tolist()

In [27]:
index = 0
for f1 in glob.glob(path_CIFs+"*.cif"):
    
        index  = index  + 1
        
        if not index == 18:
            continue 
            
        if index == 20:
            continue
            
        if index == 11:
            continue
            
        partitioned_string = f1.partition(str(MOF_type))
        f1 = partitioned_string[-1]



        # Create each MOF directory
        path = os.path.join(path_MOF_LAMMPS_NVT_MD, str(index))
        os.mkdir(path)


        # Convert supercell cif file to data.MOF file


        path_data_MOF = os.path.join(path, 'data.'+str(f1))

        path_supercell_file = os.path.join(path_MOF_NVT_MC, str(index), 'Movies', 'System_0')

        for f_m in os.listdir(path_supercell_file):
            if f_m.startswith('Framework_0_final') and f_m.endswith('_P1.cif'):

                path_supercell_CIF = os.path.join(path_supercell_file, str(f_m))

                par = Parameters(str(path_supercell_CIF))
                par.show()

                sim = LammpsSimulation(par)
                cell, graph = from_CIF(par.cif_file)
                sim.set_cell(cell)
                sim.set_graph(graph)
                sim.split_graph()
                sim.assign_force_fields()
                sim.compute_simulation_size()
                sim.merge_graphs()
                if par.output_cif:
                    #print("CIF file requested. Exiting...")
                    write_CIF(graph, cell)
                    sys.exit()

                sim.write_lammps_files(path)
                for file in os.listdir(path):
                    if file.startswith("in."):
                        os.remove(os.path.join(path, file))
                    if file.startswith("data.Framework_0_final"):
                        os.rename(os.path.join(path, file),os.path.join(path, 'data.'+str(f1)+'_O'))


        with open(os.path.join(path, 'data.'+str(f1)+'_O'),'r') as firstfile, open(str(path_data_MOF), 'a') as secondfile:    

                bc = None
                l = 0
                k = 0
                Angle_AB = None
                Angle_AC = None
                Angle_BC = None

                for line in firstfile:

                    row=line.split()

                    if len(line.strip()) == 0 :
                        if k == 1:
                            continue 
                        else:
                            secondfile.write(line)
                            continue

                    if len(row) == 3 and row[1] == 'bond' and row[2] == 'types':
                        bc = int(row[0])


                    if len(row) == 6 and row[3] == '#' :
                        l = l + 1
                        if l <= bc:
                            basic_line = '    ' + str(row[0]) +'  '+ 'harmonic'+'    ' +str(row[1])+ '        ' + str(row[2])+ '  ' + str(row[3])+ ' '+str(row[4])+' '+str(row[5]) + os.linesep
                            secondfile.write(str(basic_line)) 
                            continue
                        else:
                            continue 

                    if len(row) == 9 and row[4] == '#':   
                            basic_line = '    ' + str(row[0]) +'   '+ 'harmonic'+'     ' +str(row[1])+ '              ' + str(row[2])+ '               ' + str(row[3])+ ' '+str(row[4])+' '+str(row[5]) +' '+str(row[6])+' '+str(row[7])+' '+str(row[8]) + os.linesep
                            secondfile.write(str(basic_line))
                            continue
                    
                    
                    ### Adding angle coeff type 
                    
                    if len(row) == 8 and row[4] == '#':   
                            basic_line = '    ' + str(row[0]) +'   '+ 'cosine/periodic'+'     ' +str(row[1])+ '              ' + str(row[2])+ '               ' + str(row[3])+ ' '+str(row[4])+' '+str(row[5]) +' '+str(row[6])+' '+str(row[7]) + os.linesep
                            secondfile.write(str(basic_line))
                            continue

                    if len(row) == 11 and row[6] == '#':
                            basic_line = '    ' + str(row[0]) +'   '+ 'fourier'+'     ' +str(row[1])+ '        ' + str(row[2])+ '	   ' + str(row[3])+ '        '+str(row[4])+'               '+str(row[5]) +' '+str(row[6])+' '+str(row[7])+' '+str(row[8])+' '+str(row[9])+' '+str(row[10]) + os.linesep
                            secondfile.write(str(basic_line)) 
                            continue 

                    if len(row) == 2 and row[0] == 'Pair' and row[1] == 'Coeffs':
                            k = 1
                            continue

                    if len(row) == 6 and row[3] == 'xy' and row[4] == 'xz' and row[5] == 'yz':
                            Angle_AB = str(row[0])
                            Angle_AC = str(row[1])
                            Angle_BC = str(row[2])


                    if len(row) == 1 and row[0] == 'Atoms':

                            basic_line = 'Atoms'+ os.linesep
                            secondfile.write(str(basic_line)) 
                            blank_line = ' ' + os.linesep
                            secondfile.write(str(blank_line)) 
                            continue

                    if len(row) == 1 and row[0] == 'Bonds':
                            blank_line = ' ' + os.linesep
                            secondfile.write(str(blank_line))
                            basic_line = 'Bonds'+ os.linesep
                            secondfile.write(str(basic_line)) 
                            secondfile.write(str(blank_line)) 
                            continue

                    if len(row) == 1 and row[0] == 'Angles':
                            blank_line = ' ' + os.linesep
                            secondfile.write(str(blank_line))
                            basic_line = 'Angles'+ os.linesep
                            secondfile.write(str(basic_line)) 
                            secondfile.write(str(blank_line)) 
                            continue

                    if len(row) == 1 and row[0] == 'Dihedrals':
                            blank_line = ' ' + os.linesep
                            secondfile.write(str(blank_line))
                            basic_line = 'Dihedrals'+ os.linesep
                            secondfile.write(str(basic_line)) 
                            secondfile.write(str(blank_line)) 
                            continue

                    if len(row) == 1 and row[0] == 'Impropers':
                            blank_line = ' ' + os.linesep
                            secondfile.write(str(blank_line))
                            basic_line = 'Impropers'+ os.linesep
                            secondfile.write(str(basic_line)) 
                            secondfile.write(str(blank_line)) 
                            continue

                    else: 
                            secondfile.write(str(line))

                os.remove(os.path.join(path, 'data.'+str(f1)+'_O'))

        # Convert restart file to data.adsorbate 

        path_data_adsorbate = os.path.join(path,'data.adsorbate')

        path_restart_file = os.path.join(path_MOF_NVT_MC, str(index), 'Restart', 'System_0')

        for filename in os.listdir(path_restart_file):

            path_restart = os.path.join(path_restart_file, str(filename))

            x_O = []
            y_O = []
            z_O = []

            x_H1 = []
            y_H1 = []
            z_H1 = []

            x_H2 = []
            y_H2 = []
            z_H2 = []

            d_H1 = []
            d_H2 = []

            D_H1 = None
            D_H2 = None

            fr = open(str(path_restart), "rt")

            lines = fr.readlines()
            for line in lines:
                row=line.split()

                if not line.strip():
                    continue

                if len(row) == 7 and row[0] == 'Component:' and row[2] =='Adsorbate':
                    No_of_adsorbates = int(row[3])
                    No_of_atoms = 3*No_of_adsorbates
                    No_of_bonds = 2*No_of_adsorbates
                    No_of_angles = No_of_adsorbates

                if len(row) == 4 and row[0] == 'cell-vector-a:':
                    a = row[1]
                if len(row) == 4 and row[0] == 'cell-vector-b:':   
                    b = row[2]
                if len(row) == 4 and row[0] == 'cell-vector-c:':   
                    c = row[3]
                if len(row) == 6 and row[0] == 'Adsorbate-atom-position:' and row[2] == str(0):
                    x_O.append(row[3]) 
                    y_O.append(row[4]) 
                    z_O.append(row[5]) 
                if len(row) == 6 and row[0] =='Adsorbate-atom-position:' and row[2] == str(1) :
                    x_H1.append(row[3]) 
                    y_H1.append(row[4]) 
                    z_H1.append(row[5]) 
                if len(row) == 6 and row[0] =='Adsorbate-atom-position:' and row[2] == str(2) :
                    x_H2.append(row[3]) 
                    y_H2.append(row[4]) 
                    z_H2.append(row[5]) 

                if len(row) == 6 and row[0] =='Adsorbate-atom-position:' and row[2] == str(3) : 

                    D_H1 = ((float(x_O[-1])-float(x_H1[-1]))**2 + (float(y_O[-1])-float(y_H1[-1]))**2 + (float(z_O[-1])-float(z_H1[-1]))**2)**0.5
                    D_H2 = ((float(x_O[-1])-float(x_H2[-1]))**2 + (float(y_O[-1])-float(y_H2[-1]))**2 + (float(z_O[-1])-float(z_H2[-1]))**2)**0.5

                    d_H1.append(D_H1)
                    d_H2.append(D_H2)

            for value in d_H1: 
                if value > 1:
                    atom_number = d_H1.index(str(value))
                    print('Wrong Distance:' +str(atom_number))

            for value in d_H2: 
                if value > 1:
                    atom_number = d_H2.index(str(value))
                    print('Wrong Distance:' +str(atom_number))

            with open(str(path_data_adsorbate), 'w') as file:

                shutil.copyfile(str(path_LAMMPS_NVT_MD_input_files)+'data.adsorbate_template', str(path_data_adsorbate))

            fin = open(str(path_data_adsorbate), 'r')
            data = fin.read()
            data = data.replace('vector_A', str(a))
            data = data.replace('vector_B', str(b))
            data = data.replace('vector_C', str(c))

            data = data.replace('Angle_AB', str(Angle_AB))
            data = data.replace('Angle_AC', str(Angle_AC))
            data = data.replace('Angle_BC', str(Angle_BC))


            data = data.replace('No_of_atoms', str(No_of_atoms))
            data = data.replace('No_of_bonds', str(No_of_bonds))
            data = data.replace('No_of_angles', str(No_of_angles))

            fin.close()

            fin = open(str(path_data_adsorbate), 'w')

            fin.write(data)

            fin.close()

            ## 

            fd = open(str(path_data_adsorbate), 'r')

            lines_d = fd.readlines()

            with open(str(path_data_adsorbate), 'a') as file:

                for line in lines_d:
                    row=line.split()
                    if not line.strip():
                         continue

                    if len(row) == 8 and row[1] == 'harmonic' and row[7] == 'H_TIP4P':

                        j = 0

                        for i in range(1, int(No_of_adsorbates)+1):
                            if j == 0 :
                                file.write('\n\nAtoms\n')

                            Atoms_line_1 = os.linesep + str(j+1) + '   ' + str(i) + '   ' + str(1) + '   ' + str(0.0000) + '   ' + str(x_O[i-1])  + '   ' + str(y_O[i-1])+ '   ' + str(z_O[i-1]) 
                            file.write(str(Atoms_line_1))
                            Atoms_line_2 = os.linesep + str(j+2) + '   ' + str(i) + '   ' + str(2) + '   ' + str(0.5242) + '   ' + str(x_H1[i-1])  + '   ' + str(y_H1[i-1])+ '   ' + str(z_H1[i-1])
                            file.write(str(Atoms_line_2))
                            Atoms_line_3 = os.linesep + str(j+3) + '   ' + str(i) + '   ' + str(2) + '   ' + str(0.5242) + '   ' + str(x_H2[i-1])  + '   ' + str(y_H2[i-1])+ '   ' + str(z_H2[i-1])
                            file.write(str(Atoms_line_3))

                            j = j + 3

                        l = 0

                        for k in range(1, int(No_of_bonds)+1, 2):
                            if k == int(No_of_bonds):
                                break 
                            if l == 0 :
                                file.write('\n\nBonds\n')

                            Bonds_line_1 = os.linesep  + str(k) + '   ' + str(1) + '   ' + str(l+1) + '   ' + str(l+2)  
                            file.write(str(Bonds_line_1))
                            p =  k + 1
                            Bonds_line_2 = os.linesep  + str(p) + '   ' + str(1) + '   ' + str(l+1) + '   ' + str(l+3) 
                            file.write(str(Bonds_line_2))

                            l = l + 3                

                        file.write('\n\nAngles\n')
                        for m in range(1, int(No_of_angles)+1):

                            Angles_line_1 = os.linesep  + str(m) + '   ' + str(1) + '   ' + str(3*m-1) + '   ' + str(3*m-2) + '   ' + str(3*m)  
                            file.write(str(Angles_line_1))

        ##### Creating npt.in file

        shutil.copy2(os.path.join(path_LAMMPS_NVT_MD_input_files,'npt.in'),path)

        for filename in os.listdir(path):

            if filename == 'data.'+str(f1):

                N_O = None 
                N_H = None
                b_W = None
                a_W = None
                F_n = None 
                F_b = None 
                F_a = None
                F_d = None
                F_i = None 
                N_F = None 
                atom_types = 0
                bonds_types = 0
                angle_types = 0
                dihedral_types = 0
                improper_types = 0
                H_defect = []
                H_index = None
                O_index = None

                epsilon = []
                sigma = []
                Label = []


                path_data_MOF = os.path.join(path, filename)
                fd = open(str(path_data_MOF), "rt")
                lines = fd.readlines()
                for line in lines:
                    row=line.split()
                    if not line.strip():
                        continue
                        
                    if len(row) == 4 and row[2] == '#' and row[3] == 'H_':
                        
                        H_index = row[0]
                        
                    if len(row) == 4 and row[2] == '#' and row[3] == 'O_3':
                        
                        O_index = row[0]
 
                    if len(row) == 7 and row[1] == '444' and row[2] == str(H_index) and 0.3 < float(row[3]) < 0.5:
                        H_defect.append(row[0])
                        H_defect_atoms = ' '.join(H_defect)

                    if len(row) == 2 and row[1] == 'atoms':
                        N_F = int(row[0])
                    if len(row) == 4 and row[2] == '#':
                        atom_types = atom_types + 1
                    if len(row) == 7 and row[1] == 'harmonic':
                        bonds_types = bonds_types + 1
                    if len(row) == 10 and row[1] =='fourier':
                        angle_types = angle_types + 1
                    if len(row) == 9 and row[1] =='cosine/periodic':
                        angle_types = angle_types + 1
                    if len(row) == 10 and row[1] =='harmonic':
                        dihedral_types = dihedral_types  + 1   
                    if len(row) == 12 and row[1] =='fourier':
                        improper_types = improper_types + 1           
                    if len(row) == 4 and row[2] == '#':
                        a_string = str(row[3])
                        partitioned_string = a_string.partition('_')
                        Element = partitioned_string[0]

                        if len(Element) > 1:
                            Z = ELEMENT.index(Element[:2])
                            Label.append(str(Element[:2]))
                        if len(Element) == 1:
                            Z = ELEMENT.index(Element)
                            Label.append(str(Element))


                        epsilon.append(float(EPSILON[Z]))
                        sigma.append(float(SIGMA[Z]))

                N_O = atom_types + 1
                N_H = N_O + 1
                b_W = bonds_types + 1
                a_W = angle_types + 1
                F_n = atom_types
                F_b = bonds_types
                F_a = angle_types
                F_d = dihedral_types
                F_i = improper_types

                epsilon.append(float(0.1550))
                sigma.append(float(3.1536))

                epsilon.append(float(0.0000))
                sigma.append(float(1.0000))   

        for file in os.listdir(path):
            if file == 'data.adsorbate':
                N_W_i = None
                N_W_f = None 
                path_data_adsorbate = os.path.join(path, file)
                fdr = open(str(path_data_adsorbate), "rt")
                lines = fdr.readlines()
                for line in lines:
                    row=line.split()
                    if not line.strip():
                        continue
                    if len(row) == 2 and row[1] == 'atoms':
                        N_W_i = N_F + 1
                        N_W_f = N_F + int(row[0])


        fin = open(str(path)+'/npt.in', "rt")

        data = fin.read()
        data = data.replace('N_O', str(N_O))
        data = data.replace('N_H', str(N_H))
        data = data.replace('b_W', str(b_W))
        data = data.replace('a_W', str(a_W))
        data = data.replace('F_n', str(F_n))
        data = data.replace('O_index', str(O_index))
        data = data.replace('F_b', str(F_b))
        data = data.replace('F_a', str(F_a))
        data = data.replace('F_d', str(F_d))
        data = data.replace('F_i', str(F_i))
        data = data.replace('MOF_Name', str(f1))
        data = data.replace('N_F', str(N_F))
        data = data.replace('N_W_i', str(N_W_i))
        data = data.replace('N_W_f', str(N_W_f))
        data = data.replace('H_defect_atoms', str(H_defect_atoms))
        Label.append('O')
        Label.append('H')
        Label_atoms = ' '.join(Label)
        data = data.replace('LABEL', str(Label_atoms))

        fin.close()

        fin = open(str(path)+'/npt.in', "wt")

        fin.write(data)

        fin.close()

        #### Creating data.pair file 

        path_data_pair = os.path.join(path,'data.pair')

        with open(str(path_data_pair), 'a') as file:
            for i in range(1, int(len(epsilon)) + 1):
                for j in range(i, int(len(epsilon)) + 1):
                    pair_epsilon = float(math.sqrt(epsilon[i-1]*epsilon[j-1]))
                    pair_sigma = float(sigma[i-1]+sigma[j-1])/2
                    pair_line = 'pair_coeff' + '    ' + str(i) + '    ' + str(j) + '    ' +  'lj/cut/tip4p/long' + '    ' + str(pair_epsilon) + '    ' + str(pair_sigma) + os.linesep 
                    file.write(str(pair_line))                   

cif_file       : /storage/home/hcoda1/5/sjamdade3/scratch/August_13_2023/water_screening/MD_MC/Manuscript_Final/Defects/MOFs/defects/NVT_MC/11/Movies/System_0/Framework_0_final_2_2_2_P1.cif
output_cif     : False
output_raspa   : False
force_field    : UFF
mol_ff         : False
h_bonding      : False
dreid_bond_type: harmonic
fix_metal      : False
minimize       : False
bulk_moduli    : False
thermal_scaling: False
npt            : False
nvt            : False
cutoff         : 10.0
replication    : None
orthogonalize  : False
random_vel     : True
dump_dcd       : 0
dump_xyz       : 0
dump_lammpstrj : 0
restart        : False
tol            : 0.4
neighbour_size : 5
iter_count     : 10
max_dev        : 0.01
temp           : 298.0
pressure       : 0.00740192
nprodstp       : 200000
neqstp         : 200000
insert_molecule: 
deposit        : 0
No bonds reported in cif file - computing bonding..
totatomlen = 800
compute_topology_information()
func: cartesian_coordinates; Elps. 0.008s
func

TypeError: 'bool' object is not subscriptable

In [ ]:
shutil.copy(path_LAMMPS_NVT_MD_input_files+'Job_Array.sh', path_MOF_LAMMPS_NVT_MD)